# 07 · matplotlib 製圖（進階）

把基礎繪圖延伸到立體空間、色彩系統、複雜版面、動畫與圖片內嵌輸出，做出能對外發布的進階圖表。

## 學習目標
- 能用 3D 座標軸（3D axes）畫出立體散點、立體長條與曲面，並調整觀看視角。
- 理解色彩映射（colormap）與正規化（normalization）的關係，能讓多張圖共用同一條色階。
- 能設計多子圖（subplots）網格版面，善用軸共享（sharex / sharey）與迴圈批次填圖。
- 能用動畫（animation）產生逐格畫面並輸出成 GIF 檔。
- 能把圖表轉成記憶體位元組流（in-memory bytes），編碼成 base64 內嵌到 HTML，並用影像庫做縮圖。

In [ ]:
# 中文字型設定（本書會畫圖才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 3D 繪圖基礎（3D plotting）

3D 繪圖是在帶有第三個維度（z 軸）的座標軸上作圖，比 2D 多了深度與觀看視角（view angle）。
當資料本身有三個量（例如平面位置加高度），用立體圖能一眼看出空間分布。

與 2D 的關鍵差異：
- 軸物件不同：用 `add_subplot(projection="3d")` 取得 3D 軸（Axes3D）。
- 多了視角控制：`view_init(elev, azim)` 設定仰角（elevation）與方位角（azimuth）。

形狀：
```
ax = fig.add_subplot(projection="3d")
ax.scatter(x, y, z)
ax.view_init(elev=仰角, azim=方位角)
```

In [ ]:
# 概念：3D 散點與立體長條，並用兩種視角輸出對比
import numpy as np

rng = np.random.default_rng(0)        # 固定亂數種子，讓示範可重現
x = rng.uniform(0, 10, 40)            # 自造 40 個點的平面 x 座標
y = rng.uniform(0, 10, 40)            # 自造平面 y 座標
z = rng.uniform(0, 5, 40)            # 自造高度當作 z 軸

fig = plt.figure(figsize=(10, 4))
for i, (elev, azim) in enumerate([(20, 30), (60, 120)]):   # 兩組視角
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")      # 開立體軸
    ax.scatter(x, y, z, c=z, cmap="viridis")               # 依高度上色的 3D 散點
    ax.view_init(elev=elev, azim=azim)                     # 設定仰角與方位角
    ax.set_title(f"視角 elev={elev}, azim={azim}")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.tight_layout()
plt.show()

# 概念：立體長條圖 bar3d，需給每根長條的底座位置與長寬高
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(projection="3d")
gx, gy = np.meshgrid(range(3), range(3))      # 造 3x3 網格底座座標
gx = gx.ravel(); gy = gy.ravel()             # 攤平成一維，bar3d 要一維輸入
heights = rng.uniform(1, 6, gx.size)          # 每根長條的高度（仿真資料）
# 注意：bar3d(x, y, z, dx, dy, dz)，z 是底座高度（這裡都從 0 起算）
ax.bar3d(gx, gy, np.zeros_like(heights), 0.6, 0.6, heights, shade=True)
ax.set_title("立體長條圖 bar3d")
plt.show()

## 色彩映射與正規化（colormap & normalization）

色彩映射（colormap）是一張「數值到顏色」的對照表；數值要先經過正規化（normalization）映到 0 到 1，才能查表取色。
搞懂這層關係，顏色才能正確表達資料大小，而不是隨意上色。

兩種對應方式：

| 方式 | 物件 | 適用 |
|---|---|---|
| 連續 | `Normalize` | 數值連續變化、看漸層 |
| 離散分級 | `BoundaryNorm` + `ListedColormap` | 想把數值切成幾個級距、看分組 |

In [ ]:
# 概念：同一份數值，連續 colormap 對比離散 BoundaryNorm
from matplotlib.colors import Normalize, BoundaryNorm, ListedColormap

data = np.linspace(0, 100, 100).reshape(10, 10)   # 自造 0 到 100 的方陣當示範值

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# 連續：Normalize 把資料線性壓到 0..1，再查 viridis 色票
im0 = axes[0].imshow(data, cmap="viridis", norm=Normalize(vmin=0, vmax=100))
axes[0].set_title("連續 colormap")
fig.colorbar(im0, ax=axes[0])

# 離散：用邊界把數值切成四級，每級一個固定顏色
bounds = [0, 25, 50, 75, 100]                       # 四個級距的邊界
discrete = ListedColormap(["#ffffcc", "#a1dab4", "#41b6c4", "#225ea8"])
bnorm = BoundaryNorm(bounds, discrete.N)            # 依邊界把值歸到某一級
im1 = axes[1].imshow(data, cmap=discrete, norm=bnorm)
axes[1].set_title("離散 BoundaryNorm")
fig.colorbar(im1, ax=axes[1], ticks=bounds)
plt.tight_layout()
plt.show()
# 技巧：離散分級適合做「低中高」這種等級圖，讀者更容易看出群組
print("連續看漸層，離散看分組")

## 色條與共用色階（colorbar & shared scale）

色條（colorbar）是把色彩映射畫成一條圖例，告訴讀者顏色對應什麼數值。
當多張圖要互相比較時，必須共用同一個數值範圍（shared color scale），否則同一顏色在不同圖代表不同數值，會誤導讀者。

做法：建立一個只負責「值到色」的純色彩映射物件 `ScalarMappable`，綁定統一的 `Normalize`，再用它畫出一條共用色條。

In [ ]:
# 概念：先看各自色階的誤導，再用統一 Normalize + ScalarMappable 修正
from matplotlib.cm import ScalarMappable

# 自造三組數值範圍差很多的小資料
groups = [rng.uniform(0, 10, (5, 5)),
          rng.uniform(0, 50, (5, 5)),
          rng.uniform(0, 100, (5, 5))]

# 各自色階：每張自動依自己的最小最大值上色，顏色無法互相比較
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, g in zip(axes, groups):
    im = ax.imshow(g, cmap="viridis")   # 沒給 norm，各自獨立縮放
    fig.colorbar(im, ax=ax)
fig.suptitle("各自色階：同色不同義（誤導）")
plt.tight_layout(); plt.show()

# 修正：用全域最小最大值建一個共用 Normalize
vmin = min(g.min() for g in groups)
vmax = max(g.max() for g in groups)
shared = Normalize(vmin=vmin, vmax=vmax)            # 三張共用同一數值範圍
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, g in zip(axes, groups):
    ax.imshow(g, cmap="viridis", norm=shared)       # 全部套同一 norm
sm = ScalarMappable(norm=shared, cmap="viridis")    # 純色彩映射物件，當共用色條來源
fig.colorbar(sm, ax=axes, label="統一數值範圍")       # 一條色條服務三格
fig.suptitle("共用色階：顏色可比較")
plt.show()
print(f"全域範圍 vmin={vmin:.1f}, vmax={vmax:.1f}")

## 多子圖排版（subplots layout）

多子圖（subplots）是在一張畫布上排出多個面板的網格，常用於把多組資料並列比較。
用迴圈批次填圖（loop filling）能系統化地把每組資料填進對應格子，避免一格一格手寫。

軸共享（sharex / sharey）讓同一排或同一列共用刻度，省去重複標示、讓對比更清楚。

形狀：
```
fig, axes = plt.subplots(列數, 行數, sharex=True)
for ax, 資料 in zip(axes.ravel(), 資料群):
    ax.plot(...)
```

In [ ]:
# 概念：用迴圈把多組同結構序列填進 2xN 網格，並共享 x 軸
t = np.linspace(0, 2 * np.pi, 100)      # 共用的時間軸
n = 3                                    # 每排三格
# 自造 6 組同結構序列：不同頻率與振幅的正弦波
series = [("freq=%d" % f, a * np.sin(f * t)) for f, a in
          [(1, 1), (2, 1), (3, 1), (1, 2), (2, 2), (3, 2)]]

fig, axes = plt.subplots(2, n, figsize=(11, 5), sharex=True)   # 上下兩排共享 x 軸
for ax, (label, ys) in zip(axes.ravel(), series):              # ravel 把 2D 軸陣列攤平成一維好迴圈
    ax.plot(t, ys)
    ax.set_title(label)
    ax.axhline(0, color="gray", lw=0.5)
# 注意：sharex 之下只有最底排會顯示 x 刻度，自動省去上排重複標示
fig.suptitle("2x3 子圖網格（共享 x 軸）")
plt.tight_layout()
plt.show()
print(f"共填入 {len(series)} 個子圖")

## 動畫與 GIF 輸出（animation & GIF）

動畫（animation）是「同一張圖反覆更新內容」的逐格產物；每一格稱為影格（frame）。
`FuncAnimation` 會對每個影格呼叫一個更新函式（update function），由它改變圖上的資料。

理解更新函式如何隨影格改變資料，就能把靜態圖變成動態 GIF，最後用 `PillowWriter` 寫出 GIF 檔。

In [ ]:
# 概念：FuncAnimation 逐格更新正弦波，PillowWriter 輸出 GIF
from matplotlib.animation import FuncAnimation, PillowWriter

xs = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(6, 3))
line, = ax.plot(xs, np.sin(xs))         # 先畫第 0 格，line 物件之後反覆更新
ax.set_ylim(-1.1, 1.1)
ax.set_title("隨時間推進的正弦波")

def update(frame):                       # 更新函式：frame 是目前影格編號
    line.set_ydata(np.sin(xs + frame * 0.2))   # 相位隨影格平移，看起來像在動
    return (line,)                       # 回傳被改動的物件

# 注意：frames 決定總影格數，interval 是每格毫秒數
anim = FuncAnimation(fig, update, frames=30, interval=80, blit=True)
out_gif = "wave.gif"
anim.save(out_gif, writer=PillowWriter(fps=12))   # 用 Pillow 寫出 GIF
plt.close(fig)                            # 關掉避免在 notebook 多印一張靜態圖
print("已輸出 GIF：", out_gif, "存在嗎：", os.path.exists(out_gif))

## 圖內嵌與影像處理（embed & image）

把圖表不落地存檔，而是直接存進記憶體位元組流（in-memory bytes），再做 base64 編碼（base64 encoding）內嵌到 HTML，是報表與儀表板常見手法。
之後可用影像庫 PIL 做縮圖（thumbnail）控制檔案大小。

流程：
1. `savefig(BytesIO)` 把圖存進記憶體緩衝。
2. `base64` 把位元組編成可放進 HTML 的字串。
3. 組成 `<img src="data:image/png;base64,...">` 標籤。
4. 用 PIL 開緩衝、`thumbnail` 縮小尺寸。

In [ ]:
# 概念：圖存進 BytesIO，編 base64 組 <img>，再用 PIL 縮圖
import base64
from io import BytesIO
from PIL import Image

# 自造一張折線圖
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(np.arange(10), np.cumsum(rng.normal(size=10)))   # 累積隨機漫步當示範線
ax.set_title("待內嵌的折線圖")

buf = BytesIO()                           # 記憶體位元組流，當虛擬檔案
fig.savefig(buf, format="png", dpi=80)    # 把圖寫進緩衝而非磁碟
plt.close(fig)
buf.seek(0)                               # 寫完後把讀取位置移回開頭
png_bytes = buf.getvalue()

b64 = base64.b64encode(png_bytes).decode("ascii")   # 位元組編成純文字
img_tag = f'<img src="data:image/png;base64,{b64}">'  # 可直接貼進 HTML 的標籤
print("img 標籤開頭：", img_tag[:60], "...")

# 技巧：用 PIL 開同一份位元組流做縮圖，控制輸出大小
buf.seek(0)
im = Image.open(buf)
print("原始尺寸：", im.size)
im.thumbnail((120, 120))                  # 等比縮到最長邊不超過 120 像素
print("縮圖尺寸：", im.size)

## 練習

以下三題由淺到深，皆為複合型但簡短，資料一律用 numpy / list 自造（仿真即可），完成後請執行確認輸出。

In [ ]:
# TODO 1 ·（簡單）建築群立體散點上色（整合：3D 繪圖 + 色彩映射）
#   情境：把一小批建築用平面位置與樓高畫成立體散點，並依容積率（FAR, floor area ratio）上色。
#   要求：
#     1. 用 numpy 自造 30 棟建築的 x、y（平面座標）、height（樓高，當 z 軸）與 far（容積率，仿真數字即可）。
#     2. 用 add_subplot(projection="3d") 開立體軸，畫 3D 散點，把 x, y, height 當座標。
#     3. 把 c=far 配上連續 colormap（如 viridis）為每個點上色，並加一條色條。
#   驗收：一張立體散點圖，較高的點分布在上方，顏色由低 FAR 到高 FAR 連續變化。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）多視角建築面板共用色條（整合：3D 繪圖 + 多子圖排版 + 迴圈批次填圖 + 色條共用）
#   情境：把同一批街廓（block）建築用三個不同視角各畫一格，做成一頁三面板報告，三格共用同一色階。
#   要求：
#     1. 用 numpy 自造一批建築的 x、y、height 與 gfa（樓地板面積 gross floor area）。
#     2. 用 subplots 搭配 subplot_kw={"projection":"3d"} 開 1x3 立體子圖，用迴圈把同一份資料填進三格。
#     3. 在迴圈中對每格呼叫 view_init 給不同仰角 / 方位角，並用同一個 Normalize（依 gfa 全域最小最大值）上色一致。
#     4. 用 ScalarMappable 搭配該 Normalize 為整張圖加一條共用色條。
#   驗收：三個視角的立體散點並排，三格顏色對應同一數值範圍，右側一條共用色條同時服務三格。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）政策情境前後的網格容積聚合對比
#   （整合：多子圖排版 + 軸共享 + 離散色彩 BoundaryNorm+ListedColormap + 色條 + base64 內嵌）
#   情境：把建築依平面位置聚合到方格（grid cell），比較「現況」與「套高度乘數後」兩情境各網格的總樓地板面積分級，
#         並把結果圖內嵌成 HTML 字串。
#   要求：
#     1. 用 numpy 自造建築的 x、y、footprint（占地面積）與 floors（樓層數），令 gfa = footprint * floors。
#     2. 把 x, y 切成 NxN 網格，聚合落在同一格的建築 gfa 加總得現況容積矩陣；對 floors 乘高度乘數重算得政策後矩陣。
#     3. 用 subplots(1, 2, sharex=True, sharey=True) 並列兩矩陣，用同一組 BoundaryNorm + ListedColormap 做離散分級
#        （低 / 中 / 高 / 極高四級），兩圖共用一條離散色條。
#     4. 用 savefig(BytesIO) 把整張圖存進記憶體位元組流，編 base64 組成 <img> 字串並印出開頭片段。
#   驗收：左右兩格網格熱區並排、軸刻度共享、同一套離散色階分級，套乘數後高等級網格明顯變多，且印出 base64 <img> 開頭。
# TODO: 在這裡完成

## 小結

- 3D 繪圖用 `add_subplot(projection="3d")` 取得立體軸，多了 z 維度與 `view_init` 視角控制。
- 顏色的本質是「數值經正規化後查色票」；連續用 `Normalize`，離散分級用 `BoundaryNorm` + `ListedColormap`。
- 多圖比較必須共用同一 `Normalize`，並用 `ScalarMappable` 畫出一條共用色條，否則同色不同義會誤導。
- 多子圖用 `subplots` 開網格、迴圈批次填圖，`sharex` / `sharey` 共享刻度讓對比更清楚。
- 動畫是逐格更新同一張圖，`FuncAnimation` 配更新函式，`PillowWriter` 輸出 GIF。
- 圖可存進 `BytesIO` 不落地，編 base64 內嵌 HTML，再用 PIL `thumbnail` 控制大小。